# 🏥 Medical Insurance Cost Prediction
This notebook demonstrates a complete end-to-end machine learning project to predict medical insurance costs based on patient demographics and health indicators.

### Workflow:
1. **Data Exploration**: Distribution analysis and correlation mapping.
2. **Preprocessing**: Handling outliers, interaction features, and scaling.
3. **Model Selection**: Comparing multiple regression models.
4. **Optimization**: XGBoost hyperparameter tuning.
5. **Inference**: Interactive prediction dashboard.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_theme(style="whitegrid")

## 1. Data Collection

In [ ]:
url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
df = pd.read_csv(url)
print(f"Dataset Rows: {len(df)}")
df.head()

## 2. Preprocessing & Feature Engineering

In [ ]:
# Encoding
df['sex'] = df['sex'].map({'female': 0, 'male': 1})
df['smoker'] = df['smoker'].map({'no': 0, 'yes': 1})
region_dummy = pd.get_dummies(df['region'], prefix='region')
df = pd.concat([df, region_dummy], axis=1).drop('region', axis=1)

# Interaction Features
df['bmi_smoker'] = df['bmi'] * df['smoker']
df['age_bmi'] = df['age'] * df['bmi']

X = df.drop('charges', axis=1)
y = df['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 3. Model Development (XGBoost)

In [ ]:
model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"MAE: ${mean_absolute_error(y_test, y_pred):,.2f}")

## 4. Interactive Prediction

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

w_age = widgets.IntSlider(min=18, max=100, step=1, description='Age:', value=30)
w_sex = widgets.Dropdown(options=[('Female', 0), ('Male', 1)], description='Sex:')
w_bmi = widgets.FloatSlider(min=10, max=60, step=0.1, description='BMI:', value=25)
w_child = widgets.IntSlider(min=0, max=5, description='Children:', value=0)
w_smoker = widgets.Dropdown(options=[('No', 0), ('Yes', 1)], description='Smoker:')
w_region = widgets.Dropdown(options=['northeast', 'northwest', 'southeast', 'southwest'], description='Region:')

btn = widgets.Button(description="Predict Cost", button_style='info')
output = widgets.Output()

display(w_age, w_sex, w_bmi, w_child, w_smoker, w_region, btn, output)

def on_click(b):
    with output:
        clear_output()
        # Prepare input
        reg_val = w_region.value
        input_data = {
            'age': w_age.value,
            'sex': w_sex.value,
            'bmi': w_bmi.value,
            'children': w_child.value,
            'smoker': w_smoker.value,
            'region_northeast': 1 if reg_val == 'northeast' else 0,
            'region_northwest': 1 if reg_val == 'northwest' else 0,
            'region_southeast': 1 if reg_val == 'southeast' else 0,
            'region_southwest': 1 if reg_val == 'southwest' else 0,
            'bmi_smoker': w_bmi.value * w_smoker.value,
            'age_bmi': w_age.value * w_bmi.value
        }
        
        input_df = pd.DataFrame([input_data])
        # Ensure column order matches training
        input_df = input_df[X.columns]
        scaled = scaler.transform(input_df)
        result = model.predict(scaled)[0]
        
        print(f"Estimated Insurance Cost: ${max(0, result):,.2f}")

btn.on_click(on_click)